# Entanglement Optimization: Standard Protocol on TPU

This notebook implements the full `standard_protocol` (entanglement generation followed by relaxation) using JAX on TPU.

## Instructions
1.  Go to **Runtime -> Change runtime type**.
2.  Select **Hardware accelerator: TPU v2** (or latest available).
3.  Run all cells.

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, grad, lax, random
from functools import partial
import matplotlib.pyplot as plt
import time
import numpy as np

print("JAX Version:", jax.__version__)
try:
    print("Available Devices:", jax.devices())
except RuntimeError:
    print("No TPU/GPU found. Please enable it in Runtime settings.")

## 1. Geometric Kernels
These are the core functions for calculating distances and linking numbers.

In [ ]:
def fixbound(num):
    """Ensure the number is within the bounds [0, 1]."""
    return jnp.clip(num, 0.0, 1.0)

@jit
def dist_lin_seg(point1s, point1e, point2s, point2e):
    """Calculate the shortest distance between two line segments using JAX with cond."""
    d1 = point1e - point1s
    d2 = point2e - point2s
    d12 = point2s - point1s

    D1 = jnp.dot(d1, d1)
    D2 = jnp.dot(d2, d2)
    S1 = jnp.dot(d1, d12)
    S2 = jnp.dot(d2, d12)
    R = jnp.dot(d1, d2)

    den = D1 * D2 - R**2
    
    def case1():
        (t,u) = lax.cond( D1 != 0. , 
                    lambda _: (fixbound(S1/D1), 0.),
                    lambda _: lax.cond(D2 != 0.,
                             lambda _: (0., fixbound(-S2/D2)),
                             lambda _: (0., 0.),
                             None),
                    None)        
        return (t,u)
    
    def case2_1():
        t = 0.
        u = -S2/D2
        uf = fixbound(u)
        
        (t,u) = lax.cond(uf != u, 
                    lambda _: (fixbound((uf * R + S1) / D1), uf),
                    lambda _: (t, u),
                    None)
        return (t,u)
    
    def case2_2():
        t = fixbound((S1 * D2 - S2 * R) / den)
        u = (t * R - S2) / D2
        uf = fixbound(u)
        
        (t,u) = lax.cond(uf != u, 
                    lambda _: (fixbound((uf * R + S1) / D1), uf),
                    lambda _: (t, u),
                    None)
        return (t,u)        
    
    def case2():
        (t,u) = lax.cond( den == 0. , 
                    lambda _: case2_1(),                    
                    lambda _: case2_2(),
                    None)        
        return (t,u)
    
    (t,u) = lax.cond( (D1 == 0.) & (D2 == 0.),
                        lambda _: case1(),
                        lambda _: case2(),
                        None)
    
    dist = jnp.linalg.norm(d1 * t - d2 * u - d12)
    return dist

@jit
def compute_linking_number_cartesian(p_i, p_ii, p_j, p_jj):
    r_ij = p_i - p_j
    r_ijj = p_i - p_jj
    r_iij = p_ii - p_j
    r_iijj = p_ii - p_jj

    tol = 1e-6
    n1 = jnp.cross(r_ij, r_ijj)
    n1 = n1/(jnp.linalg.norm(n1)+tol)
    n2 = jnp.cross(r_ijj, r_iijj)
    n2 = n2/(jnp.linalg.norm(n2)+tol)
    n3 = jnp.cross(r_iijj, r_iij)
    n3 = n3/(jnp.linalg.norm(n3)+tol)
    n4 = jnp.cross(r_iij, r_ij)
    n4 = n4/(jnp.linalg.norm(n4)+tol)
    
    tol_clip = 0.
    val = -1.0/(4.0*jnp.pi)*jnp.abs(jnp.arcsin(jnp.clip(jnp.dot(n1,n2),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n2,n3),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n3,n4),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n4,n1),-1.+tol_clip,1.-tol_clip)))
    return val

@jit
def compute_linking_number(x_i, y_i, z_i, phi_i, theta_i, x_j, y_j, z_j, phi_j, theta_j, l):
    p_i = jnp.array([x_i, y_i, z_i])
    p_j = jnp.array([x_j, y_j, z_j])
    u_i = jnp.array([jnp.sin(phi_i)*jnp.cos(theta_i), jnp.sin(phi_i)*jnp.sin(theta_i), jnp.cos(phi_i)])
    u_j = jnp.array([jnp.sin(phi_j)*jnp.cos(theta_j), jnp.sin(phi_j)*jnp.sin(theta_j), jnp.cos(phi_j)])

    p_ii = p_i + l*u_i
    p_jj = p_j + l*u_j
    return compute_linking_number_cartesian(p_i, p_ii, p_j, p_jj)

## 2. Helpers and Potentials
Definitions for `create_pairs`, `effective_potential`, etc.

In [ ]:
def create_pairs(m):
    N, M = m.shape
    i, j = jnp.triu_indices(N, k=1)
    m_i = m[i]
    m_j = m[j]
    m_pairs = jnp.concatenate([m_i, m_j], axis=1)
    return m_pairs

@jit
def collision_penalized_entanglement_potential(q):
    x_i, y_i, z_i, phi_i, theta_i = q[0:5]
    x_j, y_j, z_j, phi_j, theta_j = q[5:10]

    # Linking number potential
    eff_pot = compute_linking_number(x_i, y_i, z_i, phi_i, theta_i, x_j, y_j, z_j, phi_j, theta_j, 1)
    return eff_pot

@jit
def total_effective_potential(q):
    q = jnp.reshape(q, (-1, 5))
    q_pairs = create_pairs(q)
    
    def body_fun(carry, q_pair):
        return carry + collision_penalized_entanglement_potential(q_pair), None
    
    total, _ = lax.scan(body_fun, 0., q_pairs)
    return total

@jit
def simple_harmonic_line_jump(q, params):
    col_rad = params["col_rad"]
    amp = params["amp"]
    
    x_i, y_i, z_i, phi_i, theta_i = q[0:5]
    x_j, y_j, z_j, phi_j, theta_j = q[5:10]

    p_i = jnp.array([x_i, y_i, z_i])
    p_j = jnp.array([x_j, y_j, z_j])
    u_i = jnp.array([jnp.sin(phi_i)*jnp.cos(theta_i), jnp.sin(phi_i)*jnp.sin(theta_i), jnp.cos(phi_i)])
    u_j = jnp.array([jnp.sin(phi_j)*jnp.cos(theta_j), jnp.sin(phi_j)*jnp.sin(theta_j), jnp.cos(phi_j)])

    l = 1.0
    p_ii = p_i + l*u_i
    p_jj = p_j + l*u_j

    dist = dist_lin_seg(p_i, p_ii, p_j, p_jj)
    
    dist_cont = lax.cond(dist < (col_rad*2),
                         lambda _: amp*(dist-col_rad*2)**2,
                         lambda _: 0.,
                         None)
    return dist_cont

@jit
def total_harmonic_line(q, params):
    q = jnp.reshape(q, (-1, 5))
    q_pairs = create_pairs(q)
    
    def body_fun(carry, q_pair):
        return carry + simple_harmonic_line_jump(q_pair, params), None
        
    total, _ = lax.scan(body_fun, 0., q_pairs)
    return total

## 3. Initialization Logic

In [ ]:
def sph2cart(theta, phi, r=1):
    x = r * jnp.sin(theta) * jnp.cos(phi)
    y = r * jnp.sin(theta) * jnp.sin(phi)
    z = r * jnp.cos(theta)
    return jnp.stack([x, y, z], axis=-1)

def q_to_x(q):
    q = q.reshape((-1, 5))
    x0 = q[:, :3]
    # Note: original code uses phi, theta for sph2cart BUT argument order in protocols.py might be tricky.
    # Here simple check: usually spherical theta is angle with z, phi is azimuth.
    # In protocol we have q=[x,y,z,phi,theta]. 
    # sph2cart typically takes (theta, phi) or (azimuth, elevation).
    # Based on potentials.py: u_i = [sin(phi)*cos(theta), sin(phi)*sin(theta), cos(phi)]
    # This implies phi is polar angle (from z) and theta is azimutal.
    # so z = cos(phi).
    # Our helper sph2cart does: z = r * cos(theta). So we need to match args.
    # We will just implement inline transform to be safe/consistent with potentials.py
    phi = q[:, 3]
    theta = q[:, 4]
    u_x = jnp.sin(phi) * jnp.cos(theta)
    u_y = jnp.sin(phi) * jnp.sin(theta)
    u_z = jnp.cos(phi)
    offsets = jnp.stack([u_x, u_y, u_z], axis=-1)
    
    x1 = x0 + offsets
    x = jnp.concatenate([x0, x1], axis=1)
    return x

def create_random_rods(num_rods, random_keys):
    # create jnp random array
    key = random.PRNGKey(random_keys[0])
    p1s = random.uniform(key, (num_rods,3), minval=-0.5, maxval=0.5)
    key = random.PRNGKey(random_keys[1])
    phi1 = random.uniform(key, (num_rods,1), minval=0., maxval=jnp.pi)
    key = random.PRNGKey(random_keys[2])
    theta1 = random.uniform(key, (num_rods,1), minval=0., maxval=2*jnp.pi)
    q0 = jnp.concatenate([p1s, phi1, theta1], axis=1)
    
    x0 = q_to_x(q0)
    center = jnp.mean(x0[:,:3], axis=0)
    q0 = q0.at[:,:3].set(q0[:,:3] - center)
    
    q0 = q0.flatten()
    return q0

## 4. FIRE Optimizer (JAX native)

In [ ]:
@partial(jit, static_argnames=['f', 'Nmax', 'tol'])
def optimize_fire_scan(q_init, f, Nmax=1000, dt=0.01, tol=1e-5):
    grad_fn = jit(grad(f))
    
    dtmax = 10 * dt
    dtmin = 0.02 * dt
    alpha0 = 0.1
    finc = 1.1
    fdec = 0.5
    fa = 0.99
    
    v_init = jnp.zeros_like(q_init)
    init_state = (q_init, v_init, alpha0, dt, 0)
    
    def body_fun(carry, i):
        q, v, alpha, dt, Npos = carry
        
        g = grad_fn(q)
        F = -g
        
        # Power
        P = jnp.dot(F, v)
        
        # Update hyperparameters
        dt = lax.cond(P > 0, 
                      lambda _: jnp.minimum(dt * finc, dtmax),
                      lambda _: jnp.maximum(dt * fdec, dtmin),
                      None)
        alpha = lax.cond(P > 0, lambda _: alpha * fa, lambda _: alpha0, None)
        Npos = lax.cond(P > 0, lambda _: Npos + 1, lambda _: 0, None)
        
        # Velocity update
        norm_v = jnp.linalg.norm(v)
        norm_F = jnp.linalg.norm(F)
        
        v = (1.0 - alpha) * v + alpha * F * norm_v / (norm_F + 1e-8)
        
        # Integration
        v = v + dt * F
        q = q + dt * v
        
        return (q, v, alpha, dt, Npos), None
        
    final_state, _ = lax.scan(body_fun, init_state, jnp.arange(Nmax))
    q_final = final_state[0]
    f_final = f(q_final)
    return q_final, f_final


## 5. Main Protocol Execution
This runs the two-phase protocol: Entanglement -> Relaxation.

In [ ]:
def standard_protocol():
    # Parameters configuration
    num_rods = 200
    AR = 20
    rod_diameter = 1.0/AR
    params = {"col_rad": rod_diameter/2, "amp": 100.0}
    random_keys = [42, 43, 44]
    dt = 0.01
    
    print("--- Standard Protocol ---")
    print(f"N={num_rods}, AR={AR}, D={rod_diameter}")
    
    # 1. Initialize Rods
    print("\n[Phase 1] Initializing Rods...")
    q0 = create_random_rods(num_rods, random_keys)
    
    # 2. Maximize Entanglement (Minimize negative linking number)
    print("\n[Phase 2] Generating Entanglement (Maximizing Linking Number)...")
    # We run fire optimization for N iterations
    t0 = time.time()
    q_entangled, e_val = optimize_fire_scan(q0, total_effective_potential, Nmax=1000, dt=dt)
    q_entangled.block_until_ready()
    print(f"Phase 2 Complete. Time: {time.time()-t0:.2f}s. Final Energy (Neg LinkNum): {e_val:.4f}")
    
    # 3. Relax Collision (Enforce non-penetration)
    print("\n[Phase 3] Relaxing Collisions (Enforcing Non-Penetration)...")
    # Define partial potential function with fixed params
    def collision_energy(q):
        return total_harmonic_line(q, params)
    
    t1 = time.time()
    q_relaxed, c_val = optimize_fire_scan(q_entangled, collision_energy, Nmax=5000, dt=dt*0.1)
    q_relaxed.block_until_ready()
    print(f"Phase 3 Complete. Time: {time.time()-t1:.2f}s. Final Collision Energy: {c_val:.6f}")
    
    # 4. Final Analysis
    print("\n--- Results ---")
    x_final = q_to_x(q_relaxed)
    # Calculate min distance
    q_reshaped = q_relaxed.reshape((-1, 5))
    all_pairs = create_pairs(q_reshaped)
    
    # Helper to compute min dist across all pairs (vectorized)
    p1s = all_pairs[:, 0]  # x_i ...
    # We need to reconstruct full segment points for distance check
    # all_pairs has shape (M, 10). [x_i ... theta_i, x_j ... theta_j]
    
    def get_seg_endpoints(row):
        x, y, z, phi, theta = row[0:5]
        p_s = jnp.array([x,y,z])
        u = jnp.array([jnp.sin(phi)*jnp.cos(theta), jnp.sin(phi)*jnp.sin(theta), jnp.cos(phi)])
        p_e = p_s + 1.0*u
        return p_s, p_e
    
    v_get_endpoints = vmap(get_seg_endpoints)
    
    p1s_start, p1s_end = v_get_endpoints(all_pairs[:, 0:5])
    p2s_start, p2s_end = v_get_endpoints(all_pairs[:, 5:10])
     
    dists = vmap(dist_lin_seg)(p1s_start, p1s_end, p2s_start, p2s_end)
    min_d = jnp.min(dists)
    
    print(f"Minimum Distance: {min_d:.6f}")
    print(f"Target Diameter: {rod_diameter:.6f}")
    if min_d >= rod_diameter * 0.99:
        print("Constraints Satisfied: YES")
    else:
        print(f"Constraints Satisfied: NO (Violation: {rod_diameter - min_d:.6f})")
        
    return q_relaxed

# Run it!
q_final = standard_protocol()

## 6. Visualization
Simple 3D plot of the final configuration.

In [ ]:
def plot_rods(q):
    x = q_to_x(q)
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    x0 = x[:, :3]
    x1 = x[:, 3:]
    
    # Plot lines
    for i in range(x.shape[0]):
        ax.plot([x0[i,0], x1[i,0]], [x0[i,1], x1[i,1]], [x0[i,2], x1[i,2]], 'b-', alpha=0.6)
        
    # Make axes equal
    limits = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
    origin = np.mean(limits, axis=1)
    radius = 0.5 * np.max(np.abs(limits[:, 1] - limits[:, 0]))
    ax.set_xlim3d([origin[0] - radius, origin[0] + radius])
    ax.set_ylim3d([origin[1] - radius, origin[1] + radius])
    ax.set_zlim3d([origin[2] - radius, origin[2] + radius])
    plt.show()

plot_rods(q_final)